# Wide-model validation: NVIDIA Nemotron ColEmbed 4B (2,560-d) on ViDoRe

One session = one split: smoke-test the model (first session only), encode the
500 pages and their queries, then run the whole analysis **here** and export only
a small results zip. The vectors (~10 GB per split) never leave Kaggle.

**Before running**
1. Set `SPLIT` in the parameters cell (`"docvqa"` first, then `"infovqa"` in a
   second session with `RUN_SMOKE = False`).
2. *Settings -> Accelerator*: **GPU T4 x2**. *Internet*: **On**.
3. *Save Version -> Save & Run All (Commit)*. About 4-6 hours.
4. Download `wide_results_<split>.zip` from the *Output* tab (a few MB).

Model licence: CC-BY-NC-4.0 (non-commercial research use). The weights are only
downloaded into this session.

Every step stops the notebook if a check fails.

In [ ]:
# parameters
MODEL = "nvidia/nemotron-colembed-vl-4b-v2"
SPLIT = "docvqa"          # "docvqa" or "infovqa"
RUN_SMOKE = True          # first session only

In [ ]:
import subprocess, sys, os, json, shutil, glob, time
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
import torch
assert torch.cuda.device_count() >= 2, "Needs Settings -> Accelerator -> GPU T4 x2 (two GPUs)"
print("torch", torch.__version__, [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
total, used, free = shutil.disk_usage("/tmp")
print(f"/tmp free: {free / 1e9:.0f} GB")
assert free > 30e9, "need > 30 GB free in /tmp for one split's vectors"
print(open("/proc/meminfo").read().splitlines()[0])

## 1. Install (the checkout must not be named `optivision`)

In [ ]:
BRANCH = "wide-model-validation"
WORK = "/kaggle/working"
REPO = f"{WORK}/optivision-rag"
DATA = "/tmp/optdata"                      # vectors stay here, outside the output
RESULTS = f"{WORK}/wide_results_{SPLIT}"
!rm -rf {REPO} && git clone -q --depth 1 --branch {BRANCH} https://github.com/Daemon-VI/optivision-rag.git {REPO}
!cd {REPO} && git log --oneline -1
!pip install -q -e "{REPO}[bench,merge]" 2>&1 | tail -2
# Kaggle preinstalls torchao 0.10; peft refuses to work while an incompatible torchao is importable
!pip uninstall -y -q torchao
!python -c "import torch, transformers, importlib.util as u; print('torch', torch.__version__, '| transformers', transformers.__version__, '| torchao present:', u.find_spec('torchao') is not None)"
os.makedirs(f"{DATA}/vectors", exist_ok=True); os.makedirs(RESULTS, exist_ok=True)

def run(cmd, log, env=None):
    t = time.time()
    with open(f"{RESULTS}/{log}", "w") as f:
        r = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT, text=True, env={**os.environ, **(env or {})})
    tail = open(f"{RESULTS}/{log}").read().splitlines()
    print("\n".join(tail[-25:]))
    print(f"-- {log}: exit {r.returncode} after {(time.time() - t) / 60:.1f} min")
    assert r.returncode == 0, f"{log} failed: see the lines above"

## 2. Smoke test (first session only)

Loads the model in float32 over both GPUs, encodes 6 pages and queries, checks
identity, width, finiteness and norms, compares our MaxSim with the model's own
`colbert_score`, then re-encodes in float16 on one GPU to measure how much that
changes (this decides whether the 8B, which only fits in float16, is worth running).

In [ ]:
if RUN_SMOKE:
    run([sys.executable, f"{REPO}/scripts/wide_smoke.py", "--model", MODEL, "--multi-gpu", "--pages", "6",
         "--compare-fp16", "--out", f"{RESULTS}/smoke.json"], "smoke.log")
    s = json.load(open(f"{RESULTS}/smoke.json"))
    print({k: s[k] for k in ("dimension", "vectors_per_page", "query_retrieves_own_page", "seconds_per_page",
                             "scorer_check", "placement")})
    print("fp16:", s.get("fp16"))
    print(f"estimated encode time for 500 pages: {500 * s['seconds_per_page'] / 60:.0f} min")

## 3. Encode the split (float32, both GPUs)

In [ ]:
DATASET = {"docvqa": "vidore/docvqa_test_subsampled", "infovqa": "vidore/infovqa_test_subsampled"}[SPLIT]
run([sys.executable, f"{REPO}/scripts/encode_vectors.py", MODEL, DATASET, "--backend", "nemotron", "--multi-gpu",
     "--out", f"{DATA}/vectors"], "encode.log")
TAG = f"{MODEL.split('/')[-1]}_{DATASET.split('/')[-1]}"
!ls -la {DATA}/vectors

## 4. Analysis, on the GPU for scoring

`s13_wide_study.py`: baseline, fixed pipelines (codec / token / dimension and their
combinations), memory, latency, two-tier rows, Pareto frontier.
`s7b_selection_rules.py --space=both`: the held-out calibration study (20 splits x
3 targets) on the default search space and on the projection-extended one, from
one scoring of the candidates.

In [ ]:
ENV = {"OPTIVISION_DATA": DATA, "OPTIVISION_OUT": RESULTS, "OPTIVISION_SCORE_DEVICE": "cuda",
       "OPTIVISION_CACHE_BYTES": str(4 * 1024**3), "PYTHONUNBUFFERED": "1"}
run([sys.executable, f"{REPO}/scripts/universal_study/s13_wide_study.py", f"vec:{TAG}"], "s13_wide.log", ENV)

In [ ]:
run([sys.executable, f"{REPO}/scripts/universal_study/s7b_selection_rules.py", f"vec:{TAG}", "--space=both"],
    "s7b_selection.log", ENV)

## 5. Package the results (small)

In [ ]:
shutil.copy(f"{DATA}/vectors/{TAG}_qrels.json", RESULTS)
!pip freeze > {RESULTS}/pip_freeze.txt
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv > {RESULTS}/gpu.txt
!cd {REPO} && git rev-parse HEAD > {RESULTS}/commit.txt
!find {RESULTS} -type f | sort
!cd {WORK} && rm -f wide_results_{SPLIT}.zip && zip -q -r wide_results_{SPLIT}.zip wide_results_{SPLIT} && ls -la wide_results_{SPLIT}.zip